# Figure 2 generation

These data correspond to preliminary tests ran using a weak UV as the CS because we thought it might be more salient to planaria. We were wary because UV itself causes contractions, so there was a risk of conflating sensitization with learning. But that's what control groups are for, so we tried it. The data here do NOT correspond to the runs done in the first week of April 2025. These are the preliminary ones that came before (20250328).

To add new data:
1) Ensure the data is in the CSV_FILEPATH and that it takes the expected format (see Google Sheet Misc_Paradigms_EXPORT)
2) In graphing code cell, add expected troupe to dictionary, where troupe is dictionary name and fill in colors, line style, etc.
3) Add troupe to graphing function call argument.

## Import packages, preprocess data

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Set file path
CSV_FILEPATH = '../hand_scored_datasheets/Conditioning_Variants_HandScored_Sc1.csv'  # Update to your path

# Read the CSV
df = pd.read_csv(CSV_FILEPATH)

# Forward fill the Troupe column to fill empty cells
df['Troupe'] = df['Troupe'].replace('', np.nan)
df['Troupe'] = df['Troupe'].fillna(method='ffill')

# Forward fill the BlockNum column to fill empty cells (NEW!)
df['BlockNum'] = df['BlockNum'].replace('', np.nan)
df['BlockNum'] = df['BlockNum'].fillna(method='ffill')

# Convert W1-W6 columns to numeric, coercing errors to NaN
worm_columns = ['W1', 'W2', 'W3', 'W4', 'W5', 'W6']
for col in worm_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert BlockNum to numeric (do this AFTER forward-filling)
df['BlockNum'] = pd.to_numeric(df['BlockNum'], errors='coerce')

print("Data loaded and preprocessed successfully!")
print(f"Total rows: {len(df)}")
print(f"Unique Troupe values: {df['Troupe'].unique()}")
print(f"Unique BlockNum values: {sorted(df['BlockNum'].dropna().unique())}")
print(f"BlockNum range: {df['BlockNum'].min()} to {df['BlockNum'].max()}")

Data loaded and preprocessed successfully!
Total rows: 900
Unique Troupe values: ['GD_test_1' 'bum_SM_test_1' 'SM_test_1' 'GD_1_UVSHOCK' 'GD_S_UVSHOCK'
 'GT_Charles_CSuvdimUCSshock' 'GD_B_WHITEUV' 'SM_WHITEUV' 'SM_PULSEUV'
 'SM_PULSEUV_oops,wrong_troupe']
Unique BlockNum values: [1.0, 2.0, 3.0, 4.0, 5.0]
BlockNum range: 1.0 to 5.0


## Define graphing code

In [14]:
import colorsys
import matplotlib.colors as mcolors

def _generate_worm_shades(base_color, n=6):
    """Generate n unique shades of base_color by varying lightness in HLS space."""
    rgb = mcolors.to_rgb(base_color)
    h, l, s = colorsys.rgb_to_hls(*rgb)
    light_lo, light_hi = 0.35, 0.80
    shades = []
    for i in range(n):
        t = i / max(n - 1, 1)
        new_l = light_lo + t * (light_hi - light_lo)
        r, g, b = colorsys.hls_to_rgb(h, new_l, s)
        shades.append((r, g, b))
    return shades


def create_graphs(data, groups=['GD_test_1', 'SM_test_1'], blocks=None, 
                  error_type='SEM', figsize=(10, 6),
                  title='Performance Across Blocks',
                  ylabel='Average Score',
                  y_values='probability',
                  ylim='auto', ylim_padding=0.1,
                  save=True, filename='block_performance.svg',
                  show_error_bars=True,
                  show_individual_worms=True):
    """
    Creates a line graph showing average performance across blocks for different groups.
    
    NOTE: This function handles variable trial counts automatically. It collects ALL trials 
    for each block and averages all W1-W6 values found.
    
    Parameters:
    -----------
    data : pandas DataFrame
        The preprocessed dataframe containing behavioral data
    groups : list of str
        List of troupe names to plot (e.g., ['GD_test_1', 'SM_test_1'])
        Default: ['GD_test_1', 'SM_test_1']
    blocks : list of int or None
        Specific blocks to include. If None, uses all available blocks
    error_type : str
        Type of error bars: 'SEM' or '95CI' (default: 'SEM')
    figsize : tuple
        Figure size (width, height) (default: (10, 6))
    title : str
        Plot title (default: 'Performance Across Blocks')
    ylabel : str
        Y-axis label (default: 'Average Score')
    y_values : str
        Type of y-axis values: 'probability' or 'counts' (default: 'probability')
        - 'probability': Average of all W1-W6 values (proportions)
        - 'counts': Sum each worm's scores, then average the 6 sums
    ylim : str, float, or tuple
        Y-axis limits. Options:
        - 'auto': Automatically determine based on data (default)
        - float: Sets max limit (min will be 0)
        - tuple (min, max): Explicitly set both limits
    ylim_padding : float
        Padding factor for auto y-limits (0.1 = 10% padding) (default: 0.1)
        Only used when ylim='auto'
    save : bool
        If True, saves the plot instead of displaying it (default: False)
    filename : str
        Filename for saving the plot (default: 'block_performance.png')
    show_error_bars : bool
        If True, displays error bars (default: True)
    show_individual_worms : bool
        If True, plots individual worm lines in unique shades of the group color (default: True)
    
    Returns:
    --------
    dict : Summary statistics for each group and block
    """
    
    # Validate error_type
    if error_type.upper() not in ['SEM', '95CI', 'CI']:
        raise ValueError("error_type must be 'SEM' or '95CI'")
    
    # Validate y_values
    if y_values.lower() not in ['probability', 'counts']:
        raise ValueError("y_values must be 'probability' or 'counts'")
    
    # Define styling for each group
    group_styles = {
        'GD_test_1': {
            'color': '#FF8C00',      # Orange (dark orange)
            'linestyle': '-',         # Solid
            'marker': 'o',            # Filled circle
            'fillstyle': 'full',
            'label': 'GD-1 test UV-Shock (n=6)'
        },
        'SM_test_1': {
            'color': '#2E7D32',       # Green (forest green)
            'linestyle': '-',         # Solid
            'marker': 'o',            # Filled circle
            'fillstyle': 'full',
            'label': 'SM test 1'
        },
        'GD_S_UVSHOCK': {
            'color': 'teal',          # #teal
            'linestyle': '-',         # Solid
            'marker': 'o',            # Filled circle
            'fillstyle': 'full',
            'label': 'GD-Spring water UV-Shock (n=6)'       
        },
        'GD_B_WHITEUV': {
            'color': '#002676',       # Berkeley blue
            'linestyle': '-',         # Solid
            'marker': 'o',            # Filled circle
            'fillstyle': 'full',
            'label': 'GD-Berkeley White-UV (n=2)'
        },
        'GD_1_UVSHOCK' : {
            'color': 'purple',        # Purple 
            'linestyle': '-',         # Solid
            'marker': 'o',            # Filled circle
            'fillstyle': 'full',
            'label': 'GD-1 Blue-UV (n=6)'
        },
        'SM_WHITEUV': {
            'color': '#A39F6D',       # Olive khaki thingy
            'linestyle': '-',         # Solid
            'marker': 'o',            # Filled circle
            'fillstyle': 'full',
            'label': 'SM White-UV (n=6)'},
        'SM_PULSEUV': {
            'color' : '#f05493',       # Dull pinkish color
            'linestyle': '-',         # Solid
            'marker': 'o',            # Filled circle
            'fillstyle': 'full',
            'label': 'SM Pusle-UV (n=5 or 6)'},
    }
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Store summary statistics
    summary_stats = {}
    
    # Store all plotted values for auto y-limit calculation
    all_means = []
    all_errors = []
    all_individual_vals = []
    
    # Track blocks with no data for debugging
    no_data_blocks = []
    
    # Process each group
    for group in groups:
        # Check if group has defined styling
        if group not in group_styles:
            print(f"Warning: No styling defined for group {group}, skipping...")
            continue
        
        style = group_styles[group]
        
        # Filter data for this group (exact match on Troupe column)
        group_data = data[data['Troupe'] == group].copy()
        
        if group_data.empty:
            print(f"Warning: No data found for group {group}")
            continue
        
        # Determine which blocks to process
        if blocks is None:
            available_blocks = sorted(group_data['BlockNum'].dropna().unique())
        else:
            available_blocks = sorted([b for b in blocks if b in group_data['BlockNum'].values])
        
        block_means = []
        block_errors = []
        valid_blocks = []
        worm_block_values = {col: [] for col in worm_columns}
        worm_valid_blocks = {col: [] for col in worm_columns}
        
        # Calculate statistics for each block
        for block_num in available_blocks:
            block_data = group_data[group_data['BlockNum'] == block_num]
            
            if block_data.empty:
                no_data_blocks.append(f"{group} Block {int(block_num)}")
                continue
            
            if y_values.lower() == 'probability':
                all_values = []
                for worm_col in worm_columns:
                    values = block_data[worm_col].dropna()
                    all_values.extend(values.tolist())
                    if len(values) > 0:
                        worm_val = values.mean()
                        worm_block_values[worm_col].append(worm_val)
                        worm_valid_blocks[worm_col].append(block_num)
                        all_individual_vals.append(worm_val)
                
                if not all_values:
                    no_data_blocks.append(f"{group} Block {int(block_num)}")
                    print(f"Warning: No valid data for {group}, Block {block_num}")
                    continue
                
                mean_val = np.mean(all_values)
                sem_val = np.std(all_values, ddof=1) / np.sqrt(len(all_values))
                
                if error_type.upper() == 'SEM':
                    error_val = sem_val
                    error_label = 'SEM'
                else:
                    n = len(all_values)
                    t_critical = stats.t.ppf(0.975, df=n-1)
                    error_val = t_critical * sem_val
                    error_label = '95% CI'
                
                n_datapoints = len(all_values)
                std_val = np.std(all_values, ddof=1)
                
            else:  # y_values == 'counts'
                worm_sums = []
                for worm_col in worm_columns:
                    values = block_data[worm_col].dropna()
                    if len(values) > 0:
                        worm_sum = values.sum()
                        worm_sums.append(worm_sum)
                        worm_block_values[worm_col].append(worm_sum)
                        worm_valid_blocks[worm_col].append(block_num)
                        all_individual_vals.append(worm_sum)
                
                if not worm_sums:
                    no_data_blocks.append(f"{group} Block {int(block_num)}")
                    print(f"Warning: No valid data for {group}, Block {block_num}")
                    continue
                
                mean_val = np.mean(worm_sums)
                sem_val = np.std(worm_sums, ddof=1) / np.sqrt(len(worm_sums))
                
                if error_type.upper() == 'SEM':
                    error_val = sem_val
                    error_label = 'SEM'
                else:
                    n = len(worm_sums)
                    t_critical = stats.t.ppf(0.975, df=n-1)
                    error_val = t_critical * sem_val
                    error_label = '95% CI'
                
                n_datapoints = len(worm_sums)
                std_val = np.std(worm_sums, ddof=1)
            
            block_means.append(mean_val)
            block_errors.append(error_val)
            valid_blocks.append(block_num)
            
            all_means.append(mean_val)
            all_errors.append(error_val)
            
            if group not in summary_stats:
                summary_stats[group] = {}
            
            summary_stats[group][f'Block_{int(block_num)}'] = {
                'mean': mean_val,
                'sem': sem_val,
                'error_value': error_val,
                'error_type': error_label,
                'n_datapoints': n_datapoints,
                'std': std_val,
                'calculation_method': y_values
            }
            
            print(f"{group} Block {int(block_num)}: Mean={mean_val:.3f}, {error_label}={error_val:.3f}, N={n_datapoints}")
        
        # Plot individual worm lines first (behind the mean)
        if show_individual_worms and valid_blocks:
            active_worms = [col for col in worm_columns if len(worm_block_values[col]) > 0]
            shades = _generate_worm_shades(style['color'], n=len(active_worms))
            for idx, worm_col in enumerate(active_worms):
                worm_label = f"{worm_col}" if idx == 0 else None
                ax.plot(worm_valid_blocks[worm_col], worm_block_values[worm_col],
                        linestyle='-', linewidth=1, color=shades[idx],
                        alpha=0.45, marker='.', markersize=4)
        
        # Plot the mean line for this group on top
        if valid_blocks:
            if show_error_bars:
                ax.errorbar(valid_blocks, block_means, yerr=block_errors,
                           marker=style['marker'], markersize=8, 
                           markerfacecolor=style['color'],
                           markeredgecolor=style['color'], markeredgewidth=1.5,
                           fillstyle=style['fillstyle'],
                           linestyle=style['linestyle'], linewidth=2.5, color=style['color'],
                           capsize=5, capthick=2, 
                           label=style['label'], alpha=0.9, zorder=5)
            else:
                ax.plot(valid_blocks, block_means,
                       marker=style['marker'], markersize=8, 
                       markerfacecolor=style['color'],
                       markeredgecolor=style['color'], markeredgewidth=1.5,
                       fillstyle=style['fillstyle'],
                       linestyle=style['linestyle'], linewidth=2.5, color=style['color'],
                       label=style['label'], alpha=0.9, zorder=5)
    
    # Set y-axis limits based on ylim parameter
    if ylim == 'auto':
        if all_means:
            max_val = max([m + e for m, e in zip(all_means, all_errors)])
            min_val = min([m - e for m, e in zip(all_means, all_errors)])
            if all_individual_vals:
                max_val = max(max_val, max(all_individual_vals))
                min_val = min(min_val, min(all_individual_vals))
            
            y_min = max(0, min_val - ylim_padding * (max_val - min_val))
            y_max = max_val + ylim_padding * (max_val - min_val)
            
            ax.set_ylim(y_min, y_max)
            print(f"\nAuto y-limits: [{y_min:.3f}, {y_max:.3f}]")
    elif isinstance(ylim, (int, float)):
        ax.set_ylim(0, ylim)
        print(f"\nY-limits set to: [0, {ylim}]")
    elif isinstance(ylim, (list, tuple)) and len(ylim) == 2:
        ax.set_ylim(ylim[0], ylim[1])
        print(f"\nY-limits set to: [{ylim[0]}, {ylim[1]}]")
    else:
        raise ValueError("ylim must be 'auto', a number, or a tuple (min, max)")
    
    # Formatting
    ax.set_xlabel('Block Number', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10, framealpha=0.9)
    
    if blocks is None:
        ax.set_xticks(range(1, 11))
    else:
        ax.set_xticks(blocks)
    
    ax.grid(True, color='gray', linestyle='-', linewidth=0.5, alpha=0.3)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    
    if no_data_blocks:
        print("\n" + "="*50)
        print("DEBUGGING: Blocks with no valid data (all NaNs):")
        for block_info in no_data_blocks:
            print(f"  - {block_info}")
        print("="*50)
    
    # Save or show
    if save:
        filepath = f"../figures/{filename}" if filename.endswith('.svg') else "../figures/{filename}.svg"
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"\nPlot saved to: {filepath}")
        #plt.close(fig)
        plt.show()
    else:
        plt.show()
    
    return summary_stats

print("Graphing function defined successfully!")

Graphing function defined successfully!


## Generate graphs, perform statistics

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARADIGM-LEVEL COMBINED ANALYSIS
#  Depends on: df, worm_columns, _compute_bf10, _bf_label, _compute_tost
#  (all defined in the main code block above)
# ══════════════════════════════════════════════════════════════════════════════

# ── Define which troupes belong to each paradigm ─────────────────────────────
# Edit this dict if your groupings change
PARADIGM_GROUPS = {
    'UV-Shock': dict(
        troupes = ['GD_test_1', 'GD_1_UVSHOCK', 'GD_S_UVSHOCK'],
        blocks  = list(range(1, 6)),    # blocks 1–5
        color   = '#C0392B',            # dark red
        label   = 'UV-Shock  (pooled, max n=18)',
    ),
    'White-UV': dict(
        troupes = ['GD_B_WHITEUV', 'SM_WHITEUV'],
        blocks  = list(range(1, 4)),    # blocks 1–3
        color   = '#2471A3',            # steel blue
        label   = 'White-UV  (pooled, max n=12)',
        # ⚠ NOTE: GD_B_WHITEUV has blocks 1–3; SM_WHITEUV only has blocks 1–2.
        # At block 3 only GD_B_WHITEUV contributes (N drops from 12 to 6).
        # Common blocks = {1,2} only → Page's L cannot run (needs ≥ 3).
    ),
    'Pulse-UV': dict(
        troupes = ['SM_PULSEUV'],
        blocks  = list(range(1, 5)),    # blocks 1–4
        color   = '#8E44AD',            # purple
        label   = 'Pulse-UV  (max n=6)',
    ),
}


# ══════════════════════════════════════════════════════════════════════════════
#  HELPER FUNCTIONS: Bayes Factor + TOST
# ══════════════════════════════════════════════════════════════════════════════

def _compute_bf10(sarr):
    """JZS Bayes Factor for a one-sample t-test (H1: mean != 0 vs H0: mean == 0).
    Uses the BayesFactor-style approximation with Cauchy prior (r = sqrt(2)/2)."""
    from scipy.special import gammaln
    from scipy.integrate import quad

    n = len(sarr)
    t_stat = np.mean(sarr) / (np.std(sarr, ddof=1) / np.sqrt(n))
    v = n - 1
    r = np.sqrt(2) / 2

    def integrand(g):
        return ((1 + n * g) ** (-0.5)
                * np.exp(-0.5 * t_stat**2 * v / ((1 + n * g) * (v + t_stat**2 / (1 + n * g)) if False else 1)  # placeholder
                ))

    # Simpler closed-form BF approximation (Rouder et al. 2009, Eq 1):
    def jzs_integrand(g):
        return ((1 + g) ** ((v - 1) / 2)
                * (1 + g * (1 - t_stat**2 / (v + t_stat**2))) ** (-(v + 1) / 2)
                * (2 * np.pi) ** (-0.5)
                * g ** (-1.5)
                * np.exp(-1 / (2 * g * r**2)))

    integral, _ = quad(jzs_integrand, 0, np.inf)
    bf10 = float(integral) if np.isfinite(integral) and integral > 0 else np.nan
    bf01 = 1.0 / bf10 if bf10 > 0 and np.isfinite(bf10) else np.nan
    return bf10, bf01


def _bf_label(bf01):
    """Interpret BF01 (evidence for null) using Jeffreys' scale."""
    if not np.isfinite(bf01):
        return "BF not computable"
    if bf01 >= 100:
        return "extreme evidence for H₀"
    if bf01 >= 30:
        return "very strong evidence for H₀"
    if bf01 >= 10:
        return "strong evidence for H₀"
    if bf01 >= 3:
        return "moderate evidence for H₀"
    if bf01 >= 1:
        return "anecdotal evidence for H₀"
    if bf01 >= 1/3:
        return "anecdotal evidence for H₁"
    if bf01 >= 1/10:
        return "moderate evidence for H₁"
    if bf01 >= 1/30:
        return "strong evidence for H₁"
    return "very strong evidence for H₁"


def _compute_tost(sarr, sesoi, alpha=0.05):
    """Two One-Sided Tests for equivalence of mean slope to zero."""
    n = len(sarr)
    mu = np.mean(sarr)
    se = np.std(sarr, ddof=1) / np.sqrt(n)
    df = n - 1

    t_lower = (mu - (-sesoi)) / se
    t_upper = (mu - sesoi) / se

    p_lower = 1 - stats.t.cdf(t_lower, df)
    p_upper = stats.t.cdf(t_upper, df)

    tost_p = max(p_lower, p_upper)

    return dict(
        t_lower=t_lower, p_lower=p_lower,
        t_upper=t_upper, p_upper=p_upper,
        tost_p=tost_p, sesoi=sesoi,
        equivalent=(tost_p < alpha),
    )


# ══════════════════════════════════════════════════════════════════════════════
#  COMBINED TREND ANALYSIS FUNCTION
# ══════════════════════════════════════════════════════════════════════════════
def analyze_combined_paradigm(data, paradigm_name, troupes, blocks,
                               y_values='counts', alpha=0.05, sesoi=1.0):
    """
    Pools all worms from multiple troupes within a paradigm and tests for
    an upward trend across blocks.

    Statistical design
    ──────────────────
    Each worm contributes exactly ONE slope (regressed within its own troupe's
    available blocks). Slopes are pooled across all troupes and tested with:
      ①  one-sample t-test  +  95 % CI on the mean slope
      ②  Bayesian t-test (JZS prior)  →  BF₀₁
      ③  TOST equivalence test  →  is the slope negligibly small?

    Limitation
    ──────────
    Worms from troupes with more blocks have slopes estimated from more data
    points (higher precision). This is a minor imbalance — noted in output.

    Parameters
    ----------
    data          : DataFrame   full preprocessed dataframe
    paradigm_name : str         name shown in printed output
    troupes       : list[str]   Troupe names to pool
    blocks        : list[int]   block numbers to include
    y_values      : str         'counts' or 'probability'
    sesoi         : float       smallest meaningful slope (contractions/block)
    """

    # ── Step 1: per-worm, per-block scores ───────────────────────────────────
    worm_data = {}   # {(troupe, worm_col): {block: score}}
    for troupe in troupes:
        gdata = data[data['Troupe'] == troupe].copy()
        if gdata.empty:
            continue
        troupe_blocks = sorted([b for b in blocks if b in gdata['BlockNum'].values])
        for wc in worm_columns:
            key = (troupe, wc)
            worm_data[key] = {}
            for blk in troupe_blocks:
                vals = gdata.loc[gdata['BlockNum'] == blk, wc].dropna()
                if len(vals):
                    worm_data[key][blk] = float(
                        vals.sum() if y_values == 'counts' else vals.mean()
                    )

    # ── Step 2: per-worm linear slopes ───────────────────────────────────────
    slopes, labels = [], []
    for (troupe, wc), blk_scores in worm_data.items():
        x = sorted(blk_scores.keys())
        y = [blk_scores[b] for b in x]
        if len(x) >= 2:
            slope, *_ = stats.linregress(x, y)
            slopes.append(slope)
            labels.append(f"{troupe}_{wc}")

    if len(slopes) < 2:
        return dict(paradigm=paradigm_name,
                    error='fewer than 2 worms had ≥ 2 data points')

    sarr    = np.array(slopes)
    n       = len(sarr)
    mu      = np.mean(sarr)
    se      = np.std(sarr, ddof=1) / np.sqrt(n)
    tc      = stats.t.ppf(0.975, df=n - 1)
    t_s, p2 = stats.ttest_1samp(sarr, 0)
    p1_up   = p2 / 2 if t_s > 0 else 1.0 - p2 / 2

    slope_res = dict(
        mean=mu, sem=se, ci95=(mu - tc * se, mu + tc * se),
        t=t_s, df=n - 1, p_two=p2, p_one_up=p1_up, n=n,
        per_worm=dict(zip(labels, slopes)),
        sig_up=(p1_up < alpha and mu > 0),
    )

    # ── Step 3: Bayes Factor ─────────────────────────────────────────────────
    try:
        bf10, bf01 = _compute_bf10(sarr)
        bf_res = dict(BF10=bf10, BF01=bf01, label=_bf_label(bf01))
    except Exception as e:
        bf_res = dict(error=str(e))

    # ── Step 4: TOST ─────────────────────────────────────────────────────────
    tost_res = _compute_tost(sarr, sesoi, alpha)

    return dict(
        paradigm=paradigm_name, troupes=troupes, blocks=blocks,
        n_worms=n, slope_test=slope_res, bf=bf_res,
        tost=tost_res,
    )


# ══════════════════════════════════════════════════════════════════════════════
#  PRINT RESULTS
# ══════════════════════════════════════════════════════════════════════════════
def print_paradigm_trend_results(trend_dict, plot_title='', sesoi=1.0):
    """Formatted console output of paradigm-level trend results."""
    W = 74
    hdr = (f"PARADIGM-LEVEL TREND ANALYSIS  ─  {plot_title}"
           if plot_title else "PARADIGM-LEVEL TREND ANALYSIS")
    print("\n" + "═" * W)
    print(f"  {hdr}")
    print(f"  Worms pooled across troupes within each paradigm")
    print(f"  H₁: upward trend (slope > 0)  |  α = 0.05  |  SESOI = ±{sesoi} cts/block")
    print("═" * W)

    for paradigm_name, res in trend_dict.items():
        if res is None or 'error' in res:
            msg = res['error'] if (res and 'error' in res) else 'no data'
            print(f"\n  {paradigm_name}: {msg}")
            continue

        print(f"\n{'─' * W}")
        print(f"  PARADIGM : {paradigm_name}")
        print(f"  Troupes  : {', '.join(res['troupes'])}")
        print(f"  Blocks   : {res['blocks']}")
        print(f"  N worms with ≥ 2 blocks of data : {res['n_worms']}")

        s  = res['slope_test']
        bf = res.get('bf', {})
        ts = res.get('tost', {})
        ci = s['ci95']

        # ① slope test
        print(f"\n  ① Per-worm slope t-test  (n = {s['n']} slopes pooled)")
        print(f"     Mean slope : {s['mean']:+.3f} ± {s['sem']:.3f} SEM")
        print(f"     95 % CI    : [{ci[0]:+.3f},  {ci[1]:+.3f}]")
        print(f"     t({s['df']}) = {s['t']:+.3f},  "
              f"p (two-sided) = {s['p_two']:.4f},  "
              f"p (one-sided ↑) = {s['p_one_up']:.4f}")
        # print individual slopes compactly
        per_worm_str = '  '.join(f"{k.split('_')[-1]}({k.rsplit('_',1)[0]})={v:+.2f}"
                                  for k, v in s['per_worm'].items())
        print(f"     Per worm : {per_worm_str}")
        if s['sig_up']:
            print("     ★  SIGNIFICANT UPWARD TREND ★")
        elif ci[0] < 0 < ci[1]:
            print("     ✗  Not significant — CI crosses zero (consistent with no trend)")
        else:
            print("     ✗  Not significant")

        # ② BF
        if 'BF01' in bf:
            print(f"\n  ② Bayesian t-test  (JZS prior, r = √2/2)")
            print(f"     BF₁₀ = {bf['BF10']:.3f}   BF₀₁ = {bf['BF01']:.3f}")
            print(f"     → {bf['label']}")
        elif 'error' in bf:
            print(f"\n  ② Bayes Factor: error — {bf['error']}")

        # ③ TOST
        if ts:
            print(f"\n  ③ TOST equivalence test  (SESOI = ±{ts['sesoi']} cts/block)")
            print(f"     t_lower = {ts['t_lower']:+.3f}   p_lower = {ts['p_lower']:.4f}   "
                  f"[H₀₁: slope ≤ −{ts['sesoi']}]")
            print(f"     t_upper = {ts['t_upper']:+.3f}   p_upper = {ts['p_upper']:.4f}   "
                  f"[H₀₂: slope ≥ +{ts['sesoi']}]")
            print(f"     TOST p = max(p_lower, p_upper) = {ts['tost_p']:.4f}  →  "
                  + ("✓ EQUIVALENT to no trend" if ts['equivalent']
                     else "✗ Inconclusive"))

        # verdict
        up_sig   = s['sig_up']
        bf01_val = bf.get('BF01')
        tost_eq  = ts.get('equivalent', False)

        print(f"\n  ▶  VERDICT for {paradigm_name}:")
        if up_sig:
            print("     Evidence FOR an upward trend.")
        elif bf01_val and bf01_val >= 10 and tost_eq:
            print("     Strong evidence AGAINST an upward trend (BF₀₁ ≥ 10 + TOST).")
        elif bf01_val and bf01_val >= 3 and tost_eq:
            print("     Moderate evidence AGAINST an upward trend (BF₀₁ ≥ 3 + TOST).")
        elif bf01_val and bf01_val >= 3:
            print("     BF favours no trend; TOST inconclusive (CI too wide).")
        elif tost_eq:
            print("     TOST confirms equivalence to no trend; BF inconclusive.")
        else:
            print("     INCONCLUSIVE — no trend detected but absence not confirmed.")
            print("     With small n, absence of significance ≠ evidence of absence.")

    print("\n" + "═" * W)
    print("  ⚠  Slopes are pooled across troupes within each paradigm.")
    print("     Worms from troupes with more blocks have slopes estimated")
    print("     from more data points (minor precision imbalance).")
    print("═" * W)


# ══════════════════════════════════════════════════════════════════════════════
#  COMBINED PARADIGM PLOT
# ══════════════════════════════════════════════════════════════════════════════
def _troupe_shades(base_color, n):
    """Generate n visually distinct shades from base_color for troupe lines."""
    import colorsys
    rgb = mcolors.to_rgb(base_color)
    h, l, s = colorsys.rgb_to_hls(*rgb)
    if n == 1:
        new_l = max(0.35, min(0.65, l))
        return [colorsys.hls_to_rgb(h, new_l, s)]
    shades = []
    for i in range(n):
        t = i / (n - 1)
        new_h = (h + 0.06 * (t - 0.5)) % 1.0
        new_l = 0.35 + t * 0.35
        r, g, b = colorsys.hls_to_rgb(new_h, new_l, max(0.4, s * 0.9))
        shades.append((r, g, b))
    return shades


def create_paradigm_graph(data, paradigm_groups_dict=None,
                          error_type='SEM', figsize=(10, 6),
                          title='Pooled paradigm-level analysis',
                          ylabel='Mean number of contractions',
                          y_values='counts',
                          ylim='auto', ylim_padding=0.1,
                          save=True, filename='../figures/Fig2A_Conditioning_Variants.svg',
                          show_error_bars=True,
                          show_individual_worms=True,
                          x_offset=0.10,
                          run_trend_test=True,
                          sesoi=1.0):
    """
    One mean +/- SEM line per paradigm, pooling ALL worms from ALL troupes
    at each block.  Optionally plots individual worm traces coloured by
    troupe (each troupe gets a unique shade of its paradigm colour).

    paradigm_groups_dict : dict or None
        Leave as None to use the module-level PARADIGM_GROUPS dict.
    """
    if paradigm_groups_dict is None:
        paradigm_groups_dict = PARADIGM_GROUPS

    fig, ax       = plt.subplots(figsize=figsize)
    summary_stats = {}
    all_means, all_errors = [], []
    all_individual_vals   = []
    no_data_blocks        = []

    pnames  = list(paradigm_groups_dict.keys())
    n_p     = len(pnames)
    offsets = np.linspace(-x_offset, x_offset, n_p) if n_p > 1 else [0.0]
    p_off   = dict(zip(pnames, offsets))

    for paradigm_name, pinfo in paradigm_groups_dict.items():
        troupes = pinfo['troupes']
        blocks  = pinfo['blocks']
        color   = pinfo['color']
        label   = pinfo['label']
        off     = p_off[paradigm_name]

        # ── Individual worm traces (plotted first so they sit behind mean) ───
        if show_individual_worms:
            troupe_colors = _troupe_shades(color, len(troupes))
            troupe_legend_added = set()
            for t_idx, troupe in enumerate(troupes):
                tc = troupe_colors[t_idx]
                gdata = data[data['Troupe'] == troupe].copy()
                if gdata.empty:
                    continue
                troupe_blocks = sorted(
                    [b for b in blocks if b in gdata['BlockNum'].values])
                for wc in worm_columns:
                    worm_x, worm_y = [], []
                    for blk in troupe_blocks:
                        vals = gdata.loc[gdata['BlockNum'] == blk, wc].dropna()
                        if len(vals):
                            score = float(
                                vals.sum() if y_values == 'counts'
                                else vals.mean())
                            worm_x.append(blk + off)
                            worm_y.append(score)
                            all_individual_vals.append(score)
                    if len(worm_x) >= 1:
                        lbl = troupe if troupe not in troupe_legend_added else None
                        ax.plot(worm_x, worm_y,
                                linestyle='-', linewidth=1.5, color=tc,
                                alpha=0.45, marker='.', markersize=4,
                                label=lbl, zorder=1)
                        if lbl:
                            troupe_legend_added.add(troupe)

        # ── Paradigm-level mean ± error ──────────────────────────────────────
        block_means, block_errors, valid_blocks = [], [], []

        for blk in blocks:
            pooled_scores = []
            for troupe in troupes:
                gdata = data[data['Troupe'] == troupe].copy()
                bdata = gdata[gdata['BlockNum'] == blk]
                if bdata.empty:
                    continue
                for wc in worm_columns:
                    vals = bdata[wc].dropna()
                    if len(vals):
                        score = float(
                            vals.sum() if y_values == 'counts' else vals.mean()
                        )
                        pooled_scores.append(score)

            if not pooled_scores:
                no_data_blocks.append(f"{paradigm_name} Block {blk}")
                continue

            n_dp     = len(pooled_scores)
            mean_val = np.mean(pooled_scores)
            sem_val  = np.std(pooled_scores, ddof=1) / np.sqrt(n_dp)

            if error_type.upper() == 'SEM':
                err_val, err_label = sem_val, 'SEM'
            else:
                t_crit             = stats.t.ppf(0.975, df=n_dp - 1)
                err_val, err_label = t_crit * sem_val, '95% CI'

            block_means.append(mean_val)
            block_errors.append(err_val)
            valid_blocks.append(blk)
            all_means.append(mean_val)
            all_errors.append(err_val)

            summary_stats.setdefault(paradigm_name, {})[f'Block_{int(blk)}'] = dict(
                mean=mean_val, sem=sem_val, error_value=err_val,
                error_type=err_label, n=n_dp,
            )
            print(f"{paradigm_name} Block {int(blk)}: "
                  f"Mean={mean_val:.3f}, {err_label}={err_val:.3f}, "
                  f"N={n_dp} worms")

        if valid_blocks:
            px = [x + off for x in valid_blocks]
            kw = dict(
                marker='o', markersize=8,
                markerfacecolor=color, markeredgecolor=color,
                markeredgewidth=1.5, fillstyle='full',
                linestyle='-', linewidth=2.5,
                color=color, label=label, alpha=0.95,
                zorder=5,
            )
            if show_error_bars:
                ax.errorbar(px, block_means, yerr=block_errors,
                            capsize=5, capthick=2, **kw)
            else:
                ax.plot(px, block_means, **kw)

    # y-axis
    if ylim == 'auto' and all_means:
        hi  = max(m + e for m, e in zip(all_means, all_errors))
        lo  = min(m - e for m, e in zip(all_means, all_errors))
        if all_individual_vals:
            hi = max(hi, max(all_individual_vals))
            lo = min(lo, min(all_individual_vals))
        pad = ylim_padding * (hi - lo)
        ax.set_ylim(max(0, lo - pad), hi + pad)
    elif isinstance(ylim, (int, float)):
        ax.set_ylim(0, ylim)
    elif isinstance(ylim, (list, tuple)) and len(ylim) == 2:
        ax.set_ylim(*ylim)
    elif ylim != 'auto':
        raise ValueError("ylim must be 'auto', a number, or (min, max)")

    ax.set_xlabel('Block Number', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel,         fontsize=12, fontweight='bold')
    ax.set_title(title,           fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10, framealpha=0.9)

    all_blks = sorted({b for pinfo in paradigm_groups_dict.values()
                       for b in pinfo['blocks']})
    ax.set_xticks(all_blks)
    ax.grid(True, color='gray', linestyle='-', linewidth=0.5, alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()

    if no_data_blocks:
        print("\nBlocks with no valid data: " + ", ".join(no_data_blocks))

    if run_trend_test:
        trend_results = {
            pname: analyze_combined_paradigm(
                data, pname,
                pinfo['troupes'], pinfo['blocks'],
                y_values=y_values, sesoi=sesoi,
            )
            for pname, pinfo in paradigm_groups_dict.items()
        }
        print_paradigm_trend_results(trend_results, plot_title=title, sesoi=sesoi)
        summary_stats['_trend_analysis'] = trend_results

    if save:
        path = (f"{filename}"
                if filename.endswith('.svg')
                else f"{filename}.svg")
        fig.savefig(path, dpi=300, bbox_inches='tight')
        print(f"Saved: {path}")
        plt.close(fig)
    else:
        plt.show()


    return summary_stats


print("Paradigm-level functions defined!")




Paradigm-level functions defined!


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════════════════════════════════
# Split into two dicts: one without Pulse-UV, one with only Pulse-UV
PARADIGM_GROUPS_MAIN = {k: v for k, v in PARADIGM_GROUPS.items() if k != 'Pulse-UV'}
PARADIGM_GROUPS_PULSE = {k: v for k, v in PARADIGM_GROUPS.items() if k == 'Pulse-UV'}

# Plot 1: UV-Shock + White-UV (original plot minus Pulse-UV)
paradigm_stats = create_paradigm_graph(
    df,
    paradigm_groups_dict = PARADIGM_GROUPS_MAIN,
    error_type  = 'SEM',
    y_values    = 'counts',
    ylim        = 9,
    title       = 'Pooled paradigm-level analysis',
    ylabel      = 'Mean number of contractions',
    x_offset    = 0.10,
    sesoi       = 1.0,
    run_trend_test = True,
    show_individual_worms = True,
    save        = True,
    filename    = '../figures/Fig2A_Conditioning_Variants.svg',
)


UV-Shock Block 1: Mean=0.944, SEM=0.328, N=18 worms
UV-Shock Block 2: Mean=1.167, SEM=0.500, N=18 worms
UV-Shock Block 3: Mean=1.056, SEM=0.439, N=18 worms
UV-Shock Block 4: Mean=1.056, SEM=0.408, N=18 worms
UV-Shock Block 5: Mean=0.667, SEM=0.323, N=18 worms
White-UV Block 1: Mean=1.500, SEM=0.544, N=12 worms
White-UV Block 2: Mean=0.500, SEM=0.195, N=12 worms
White-UV Block 3: Mean=0.167, SEM=0.167, N=6 worms

══════════════════════════════════════════════════════════════════════════
  PARADIGM-LEVEL TREND ANALYSIS  ─  Pooled paradigm-level analysis
  Worms pooled across troupes within each paradigm
  H₁: upward trend (slope > 0)  |  α = 0.05  |  SESOI = ±1.0 cts/block
══════════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────────
  PARADIGM : UV-Shock
  Troupes  : GD_test_1, GD_1_UVSHOCK, GD_S_UVSHOCK
  Blocks   : [1, 2, 3, 4, 5]
  N worms with ≥ 2 blocks of data : 18

  ① Per-worm slope t-test  

In [ ]:
# Plot 2: Pulse-UV only (separate plot)
pulse_stats = create_paradigm_graph(
    df,
    paradigm_groups_dict = PARADIGM_GROUPS_PULSE,
    error_type  = 'SEM',
    y_values    = 'counts',
    ylim        = 22,
    title       = 'Pulse-UV paradigm analysis',
    ylabel      = 'Mean number of contractions',
    x_offset    = 0.0,
    sesoi       = 1.0,
    run_trend_test = True,
    show_individual_worms = True,
    save        = True,
    filename    = '../figures/Fig2B_PulseUV.svg',
)

Pulse-UV Block 1: Mean=5.000, SEM=3.271, N=5 worms
Pulse-UV Block 2: Mean=4.800, SEM=3.089, N=5 worms
Pulse-UV Block 3: Mean=8.333, SEM=3.353, N=6 worms
Pulse-UV Block 4: Mean=4.800, SEM=3.200, N=5 worms

══════════════════════════════════════════════════════════════════════════
  PARADIGM-LEVEL TREND ANALYSIS  ─  Pulse-UV paradigm analysis
  Worms pooled across troupes within each paradigm
  H₁: upward trend (slope > 0)  |  α = 0.05  |  SESOI = ±1.0 cts/block
══════════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────────
  PARADIGM : Pulse-UV
  Troupes  : SM_PULSEUV
  Blocks   : [1, 2, 3, 4]
  N worms with ≥ 2 blocks of data : 5

  ① Per-worm slope t-test  (n = 5 slopes pooled)
     Mean slope : +0.460 ± 0.201 SEM
     95 % CI    : [-0.099,  +1.019]
     t(4) = +2.283,  p (two-sided) = 0.0845,  p (one-sided ↑) = 0.0423
     Per worm : W1(SM_PULSEUV)=+0.10  W2(SM_PULSEUV)=+1.20  W3(SM_PULSEUV)=+0.50